# 向量库和资料属性

只保存向量，即使找到相似片段，也无法说明它来自哪份资料、哪一页或谁能查看。本页先说明向量、正文与资料属性怎样配合，再用三个本地记录比较筛选前后的检索结果。

Metadata 是随正文保存的来源、日期、部门和访问级别等属性。教程附带 Chroma 库可以直接读取；本页使用手写向量，方便看清相似度与过滤的关系。


## 原理：先确定可见范围，再比较相似度

资料经过读取、清理与分块后由向量模型编码，查询也使用同一模型与相同的维度、归一化和距离约定。候选可按余弦相似度、点积或欧氏距离排序。本页计算余弦相似度，先按当前用户允许的资料属性过滤，再在可见记录中排序。

每条记录至少保存稳定编号、正文、来源、页码或标题、版本与访问范围。权限条件由可信身份系统提供，用户提交的筛选参数不能直接成为访问许可；只在结果页面隐藏无权资料，也不能保证中间检索与回答没有读到它。完整示例见[检索前过滤无权访问的资料](../4.%20检索阶段/检索前过滤无权访问的资料.ipynb)。

## 实验：三个记录的筛选前后结果

查询向量为 `[1.0, 0.0]`。先看全量排序，再只允许 `level=public` 且 `year>=2024` 的资料，观察候选变化。


In [1]:
from math import sqrt

rows = [
    {"id": "a", "text": "公开的模型评估说明", "vector": [0.9, 0.1], "level": "public", "year": 2024},
    {"id": "b", "text": "内部的模型评估记录", "vector": [1.0, 0.0], "level": "internal", "year": 2025},
    {"id": "c", "text": "公开的数据库说明", "vector": [0.1, 0.9], "level": "public", "year": 2025},
]
query = [1.0, 0.0]

def cosine(left, right):
    return sum(a * b for a, b in zip(left, right)) / (sqrt(sum(a*a for a in left)) * sqrt(sum(b*b for b in right)))

all_ranked = sorted(rows, key=lambda item: cosine(query, item["vector"]), reverse=True)
allowed = [item for item in rows if item["level"] == "public" and item["year"] >= 2024]
allowed_ranked = sorted(allowed, key=lambda item: cosine(query, item["vector"]), reverse=True)

print("不筛选的第一条：", all_ranked[0]["id"], all_ranked[0]["level"])
print("先按属性筛选后：", [(item["id"], item["level"]) for item in allowed_ranked])
assert all(item["level"] == "public" for item in allowed_ranked)

不筛选的第一条： b internal
先按属性筛选后： [('a', 'public'), ('c', 'public')]


## 结果与维护要求

不筛选时，内部记录 `b` 与查询最相似；先筛选后只返回公开记录 `a`、`c`。这说明资料范围需要参与检索过程；它是三个手写向量的计算演示，没有测试真实权限服务或向量模型质量。

更新资料时，用稳定编号替换旧片段，保持正文、向量和 metadata 来自同一版本。页码、标题与来源随命中返回，便于回答后核对；访问范围变化、删除与用户隔离也要作用于实际存储和检索。


## 扩展：把字段映射到 LangChain/Chroma

下面把前面的字段映射为 `Document.page_content` 和 `Document.metadata`。示意代码用数值 `data_level` 代替 `public/internal`，只展示字段与过滤表达式的关系；运行时需要提供项目的向量模型与存储配置。

```python
# LangChain/Chroma 的字段关系示例；向量模型和文件路径请按项目配置。
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

documents = [
    Document(
        page_content="公开的模型评估说明",
        metadata={
            "source": "handbook.pdf",
            "page": 12,
            "version": "2026-01",
            "data_level": 1,
        },
    ),
    Document(
        page_content="内部的部署说明",
        metadata={
            "source": "runbook.pdf",
            "page": 3,
            "version": "2026-01",
            "data_level": 2,
        },
    ),
]
# vector_db = Chroma.from_documents(documents, embedding=embedding)
# visible = vector_db.similarity_search(
#     "模型评估", k=3, filter={"data_level": 1}
# )
# 给回答模型的证据应保留 page/source；不要只返回向量或相似度。
```


## 数据库类型和近似搜索的背景

几种常见数据库的用途如下：

| 类型 | 更擅长的任务 |
|---|---|
| 关系型数据库 | 结构化记录、事务和精确条件查询 |
| NoSQL/键值数据库 | 半结构化资料和快速点查 |
| 列式数据库 | 按列分析和数据仓库查询 |
| 图数据库 | 实体关系、路径和网络结构 |
| 全文搜索引擎 | 词项、布尔条件、日志和全文匹配 |
| 向量数据库 | 高维向量的近邻搜索，并保存正文和 metadata |

真实系统常把全文搜索、关系库和向量库组合起来：词项检索负责精确术语，向量检索负责语义相似，关系库负责权限和事务。向量库不是“把所有数据丢进去就能回答”的黑盒，分块、模型、距离度量、过滤和版本仍然决定结果。

Exact KNN 会计算查询向量与候选向量的真实距离，适合小数据集和验证参照；ANN（Approximate Nearest Neighbor，近似最近邻）通过牺牲少量召回换取速度。HNSW 用分层小世界图导航，适合低延迟近邻查询；Annoy 使用随机树；ScaNN 用量化/分区等方式缩小候选集合；NGT、RNSG 也是图结构近邻方法。索引构建参数、更新成本、过滤是否会降低近似算法效果，都要在项目数据上测量，不能只因为某个库宣称支持 HNSW 就直接采用。

### 常见产品与组件怎样区分

下面保留常见产品和组件的技术位置，目的不是排排行榜，而是避免把检索库、数据库和托管服务混为一谈。能力会随版本变化，落地前仍应按实际版本验证。

| 代表 | 技术位置 | 适合先考察的场景 | 选型时继续验证 |
|---|---|---|---|
| [FAISS](https://faiss.ai/) | 高效稠密向量相似度搜索与聚类库，不负责完整数据库治理 | 在单机或自建服务中控制索引和检索算法 | 持久化、metadata、权限、更新和高可用要由外围系统承担 |
| [Chroma](https://docs.trychroma.com/docs/overview/introduction) | 可本地、自托管或云端使用的 AI 检索数据基础设施 | 教学、原型和希望快速把正文、metadata、向量放在一起的应用 | 并发、备份、升级、容量和部署方式 |
| [Milvus](https://milvus.io/docs/overview.md) | 可从本地形态扩展到独立或分布式部署的专用向量数据库 | 数据量、吞吐或多节点扩展成为主要约束 | 运维复杂度、索引参数、过滤和资源成本 |
| [Qdrant](https://qdrant.tech/documentation/overview/what-is-qdrant/) / [Weaviate](https://docs.weaviate.io/weaviate) | 同时保存向量、对象或 payload，并提供过滤与服务 API 的专用向量数据库 | 需要独立检索服务、metadata 过滤和清晰 API 边界 | 部署模式、多租户、备份、混合检索与升级策略 |
| [Pinecone](https://docs.pinecone.io/guides/get-started/overview) | 托管式向量数据库服务 | 希望减少自建检索基础设施和运维工作 | 数据驻留、网络、费用、配额、可迁移性和服务边界 |
| [Elasticsearch](https://www.elastic.co/docs/solutions/search/vector) / OpenSearch | 在全文搜索基础上提供向量字段和近邻检索的搜索引擎 | 已有关键词检索、过滤、聚合，并希望加入语义或混合检索 | 稀疏/稠密融合、索引资源、版本兼容和相关性调参 |

选型顺序应从需求开始：先确定数据规模、更新频率、延迟、过滤/权限、多租户、混合检索、备份和运维边界，再用同一语料与 query/qrels 比较召回、延迟和成本。产品名单本身不能替代项目评测。

## Chroma 这类嵌入式向量库的边界

Chroma 可以保存集合、正文、资料属性和向量，适合在 Python 项目中建立本地相似度检索。实际使用时要明确保存目录、集合名称、同时访问的处理方式，以及备份、删除和重建办法。只放在内存中的集合会在进程结束后丢失；复制已有数据库时，也要确认向量模型、分块方式和字段一致。
